# Assess dataset coverage

Detect class imbalance, metadata gaps, missing label-space regions, and embedding blind spots with three config-driven
presets: [`bias`](../reference/presets.md#bias), [`scope`](../reference/presets.md#scope) and
[`taxonomy`](../reference/presets.md#taxonomy).

You are a T&E engineer or data scientist who needs to verify that a dataset covers required operational conditions and
label taxonomies before model training or certification.

Run coverage assessment during early data preparation alongside {doc}`data_cleaning`. While data cleaning checks data
quality, coverage evaluates completeness across classes, metadata factors, and feature spaces. Gaps identified here
inform data collection and establish trustworthy baselines for {doc}`drift_monitoring` and {doc}`ood_detection`. See
{doc}`../concepts/Coverage` for background.

## What you will do

- Load MilitaryVehicles and filter out the Air Defense category using `Filter` to simulate missing collection
  categories.
- Run `bias` and `scope` without an extractor for a fast label and metadata pass.
- See why class counts alone do not reveal a missing category.
- Run `taxonomy` on the same source, with the dataset's taxonomy as its ontology, to name the unsampled concepts.
- Re-run `scope` with a BoVW extractor to evaluate embedding coverage and dimensional completeness.
- Read each step's output from the result, and tune the checks' thresholds.
- Export the structured coverage results to JSON.

## What you will learn

- How to configure and run the `bias`, `scope` and `taxonomy` presets with `run_task()` and `run_tasks()`.
- How coverage evaluates two axes: taxonomic representation (`taxonomy`, against an ontology) and visual variation
  (`scope`, in embedding space).
- Why count-based distributions fail to detect unsampled classes when loaders drop missing categories.
- How to distinguish genuine collection gaps from intentional ontology scope boundaries.
- How to adjust each check's thresholds for varying domain risk tolerances.

## Prerequisites

- Install `dataeval-flow`, which includes `dataeval`, `datamaite` and `pydantic`.
- Install `maite-datasets` to download MilitaryVehicles.
- Make sure you have network access for the initial dataset download.

## Lay out the data root

The data root is the folder you run this notebook from. After a first run it holds:

```text
project/
├── data/
│   ├── militaryvehicles/                    # downloaded on the first run; the export is made from it
│   └── militaryvehicles_datamaite_train/    # exported on the first run; read: its images sit under train/
└── cache/                                   # written: image statistics and embeddings, reused on a re-run
```

The notebook loads the export in Python and hands the pipeline the dataset in memory, so no configuration path points
at it. The notebook builds its pipeline in Python and reads no configuration file.

## Simulate a collection with a missing category

Operational recognition systems require predefined label taxonomies. MilitaryVehicles provides a `hierarchy` attribute
categorizing tanks, BMPs, BTRs, self-propelled artillery, air defense systems, and related vehicle types.

You will drop the Air Defense category using `Filter` to simulate a collection cycle that missed a required category.
All remaining class counts and imagery remain unmodified. See {doc}`../how_to/build_dataset_views` for view operations
such as `Filter`, `Shuffle`, and `Limit`.

In [ ]:
from pathlib import Path

import numpy as np
from dataeval.data import Filter, Limit, Shuffle, View
from datamaite import load_ic
from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

vehicles_raw = load_ic(data_root / "militaryvehicles_datamaite_train" / "train", dataset_format="huggingface_vision")
index2label = vehicles_raw.metadata.get("index2label", {})

# The category this collection cycle never captured.
AIR_DEFENSE = {"30N6E", "Iskander", "Pantsir-S1", "Rs-24"}
collected_classes = [i for i, name in index2label.items() if name not in AIR_DEFENSE]

# `Filter` enforces the gap by filtering on sample labels.
# `Shuffle` and `Limit` sample the result for faster execution.
collected = View(vehicles_raw, [Filter(classes=collected_classes), Shuffle(seed=0), Limit(1500)])

print(f"Full collection:  {len(vehicles_raw)} frames, {len(index2label)} types")
print(f"After the gap:    {len(collected)} frames, {len(collected_classes)} types")
print(f"Never collected:  {sorted(AIR_DEFENSE)}")

### Check the class distribution

Start with the obvious check. Are the classes balanced?

In [ ]:
labels = np.array([int(np.argmax(collected[i][1])) for i in range(len(collected))])
class_counts = {index2label[c]: int((labels == c).sum()) for c in sorted(set(labels.tolist()))}
ordered = sorted(class_counts.items(), key=lambda kv: -kv[1])

largest, smallest = ordered[0], ordered[-1]
ratio = largest[1] / smallest[1]
print(f"Largest class:  {largest[0]} ({largest[1]})")
print(f"Smallest class: {smallest[0]} ({smallest[1]})")
print(f"Imbalance ratio: {ratio:.2f}:1")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
names = [n for n, _ in ordered]
values = [v for _, v in ordered]
ax.bar(names, values, color="#3498db")
ax.axhline(y=float(np.mean(values)), color="gray", linestyle="--", label="mean")
ax.set_ylabel("Frame count")
ax.set_title("Class distribution of the collected data")
ax.tick_params(axis="x", rotation=60)
for tick in ax.get_xticklabels():
    tick.set_horizontalalignment("right")
ax.legend()
plt.tight_layout()
plt.show()

The sample distribution shows an imbalance ratio of 1.77:1. Count-based checks indicate a balanced dataset, but cannot
identify missing categories absent an external ontology definition.

## Run `bias` and `scope` without an extractor

You can run both presets without an extractor for a fast initial pass. Each preset's settings expand to a chain of
steps, each an evaluator, a check that judges one, or a transform. `bias` reads only labels and metadata:
[`label-health`](../reference/evaluators.md#label-health),
[`factor-summary`](../reference/evaluators.md#factor-summary), [`balance`](../reference/evaluators.md#balance),
[`diversity`](../reference/evaluators.md#diversity), [`parity`](../reference/evaluators.md#parity) and
[`factor-gaps`](../reference/combines.md#factor-gaps) (the gap analysis). Without an extractor, `scope` skips the
steps that embed the images, [`coverage`](../reference/evaluators.md#coverage) and
[`completeness`](../reference/evaluators.md#completeness), and runs
[`representation`](../reference/evaluators.md#representation) (what each class lacks of an even spread). The two run
as two tasks of one pipeline, on the same source.

You will evaluate intrinsic image factors (such as brightness, contrast, and sharpness) as metadata conditions. The
gap analysis cross-tabulates class labels against binned factors to detect whether particular vehicle classes were
imaged under limited operational conditions.

`checks` is keyed by the type of the check it sets: on `bias`, `class-imbalance` judges the label distribution, and
`factor-coverage-gaps` the gap analysis. To run coverage tasks in a container, see
{doc}`../how_to/containerized_workflows`.

In [ ]:
from dataeval_flow import run_task, run_tasks
from dataeval_flow.config import DatasetProtocolConfig, MetadataPolicyConfig, PipelineConfig, SourceConfig, TaskConfig
from dataeval_flow.workflows.bias import BiasConfig
from dataeval_flow.workflows.scope import ScopeConfig

vehicle_factors = MetadataPolicyConfig(
    name="vehicle_factors",
    # Nothing in this dataset's own per-sample metadata is a collection condition:
    # `id` is unique per frame (so it looks perfectly class-predictive) and height and
    # width describe the crop, not the scene. Ask for measured image statistics instead.
    intrinsic_factors=["visual", "pixel"],
    exclude=["id", "height", "width"],
    # Explicitly declared bins keep factor cuts consistent across runs.
    continuous_factor_bins={
        "brightness": 5,
        "contrast": 5,
        "darkness": 5,
        "entropy": 5,
        "kurtosis": 5,
        "mean": 5,
        "sharpness": 5,
        "skew": 5,
        "std": 5,
        "var": 5,
        "zeros": 5,
    },
)

bias_workflow = BiasConfig.model_validate(
    {
        "name": "bias",
        "metadata": "vehicle_factors",
        "factor-gaps": {"min_representation": 5},  # Flag class-factor-value combos with < 5 samples
        "diversity": {"method": "simpson"},
        "checks": {
            "class-imbalance": {"warning": 3.0},  # Catch moderate class imbalance
            "factor-coverage-gaps": {"warning": 2},  # Warn past 2 gaps
        },
    }
)
metadata_only_workflow = ScopeConfig(name="coverage-metadata-only")

task_bias = TaskConfig(name="vehicles-bias", workflow="bias", sources="vehicles_src")
task_metadata = TaskConfig(
    name="vehicles-coverage-metadata",
    workflow="coverage-metadata-only",
    sources="vehicles_src",
    # No extractor for metadata-only pass
)

config_metadata = PipelineConfig(
    max_processes=2,
    metadata=[vehicle_factors],
    datasets=[
        DatasetProtocolConfig(name="vehicles_collected", format="maite", dataset=collected),
    ],
    sources=[
        SourceConfig(name="vehicles_src", dataset="vehicles_collected"),
    ],
    workflows=[bias_workflow, metadata_only_workflow],
    tasks=[task_bias, task_metadata],
)

In [ ]:
results = run_tasks(config_metadata, cache_dir=Path("./cache"))
result_bias, result_metadata = results["vehicles-bias"], results["vehicles-coverage-metadata"]

### Bias report

The `bias` report gives each finding a section: Class Imbalance, Shortcut Risk, Factor Parity and Factor Coverage
Gaps. The metadata summary, balance and diversity are report sections, not findings.

In [ ]:
print(result_bias.report())

### View the HTML report

`result_bias.to_html()` renders the same report as one HTML page. It holds everything the text report holds, laid out
for reading on screen, with tables you can sort and filter. Every result has a `to_html()`. The page follows your
system's dark mode setting, so here it may not match this site's theme. See {doc}`view_html_reports` for the rest of
the page.

In [ ]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [ ]:
show(result_bias.to_html())

### Coverage report (metadata only)

The `scope` report gives the Class Shortfall. Class Coverage and Dimensional Completeness are reported as not
assessed, and the Steps table at the end says why each embedding step was skipped.

In [ ]:
print(result_metadata.report())

### Drill into each step's output

The result is a `ChainResult`. `result.steps` holds each step's output by step name, for programmatic inspection. The
`bias` run's `label-health` step's output is a count of every class the dataset declares, at 0 where it has no labels.

In [ ]:
label_health = result_bias.steps["label-health"].output.data()
print(f"Number of classes: {label_health['class_count']}")
print(f"Empty images: {label_health['empty_image_count']}")
print("\nClass distribution (five largest, five smallest):")
by_size = sorted(label_health["label_counts_per_class"].items(), key=lambda x: x[1], reverse=True)
for cls, count in by_size[:5]:
    print(f"  {cls:<26} {count}")
print("  ...")
for cls, count in by_size[-5:]:
    print(f"  {cls:<26} {count}")

In [ ]:
# Metadata gaps: evaluate if a class was only imaged under narrow conditions
gaps = result_bias.steps["factor-gaps"].output
print(f"Metadata coverage gaps: {len(gaps.gaps)}\n")

print("  Mutual information (class -> factor), five strongest:")
mi = sorted(gaps.mutual_information.items(), key=lambda x: -x[1])
for factor, value in mi[:5]:
    print(f"    {factor:<28} {value:.4f}")

print("\n  Gap details (first ten):")
for gap in gaps.gaps[:10]:
    print(
        f"    {gap.class_name:<14} x {gap.factor_name}={gap.factor_value}: "
        f"count={gap.class_count}, expected={gap.expected_count:.1f}, "
        f"deficit={gap.deficit:.0%}"
    )
if not gaps.gaps:
    print("    No metadata gaps detected.")

In [ ]:
# The class worklist: each class short of an even spread over the declared classes
worklist = result_metadata.steps["representation"].output
print(f"Total deficit: {worklist.total_deficit} labels\n")
for row in worklist.data().iter_rows(named=True):
    print(f"  {row['label']:>14}  {row['action']:<8} have {row['count']:>4}, want {row['target']:>4}")

### Interpret the findings

In this output, the imbalance ratio (1.8:1) satisfies the configured threshold of 3.0. The Class Imbalance finding
warns anyway, because four declared classes, the Air Defense types, have zero samples. `class-imbalance` takes its
ratio over the classes with labels, and a declared class with none always warns.

The gap analysis finds nothing. A factor is searched for gaps only when its mutual information with the class reaches
`mi_threshold`, 0.1 by default, and the strongest here, `contrast`, scores 0.0027. The measured image statistics
barely vary with the vehicle class in this sample, so Factor Coverage Gaps is `ok`.

Shortcut Risk and Factor Parity agree that no factor tells much about the class. No factor's mutual information passes
0.1, and no factor's Cramér's V passes 0.3. The strongest, `darkness`, scores 0.066. Factor Parity also names ten
factors whose binned tables hold cells too sparse for the chi-square p-value to be trusted, 1,500 items being spread
over 24 classes and five bins.

The worklist cell lists six classes short of an even spread over the 24 declared classes, 266 labels in all: the four
Air Defense types have 0 of a target of 62, and `BTR-70` and `T-90` have 53.

`Filter` removed samples while leaving `index2label` intact in the dataset metadata. If a dataset loader drops unused
category entries completely, count-based workflows cannot detect missing categories. To guarantee detection of missing
classes regardless of dataset metadata formatting, you should declare an ontology. See
{doc}`../how_to/declare_an_ontology` for how to define a label space with an inline dictionary or a SKOS/OWL file.

## Judge the labels against the sanctioned label space

The `taxonomy` preset compares a dataset's labels against a declared ontology, the full taxonomy specification. It
runs on the same source as `bias` and `scope`, as a task of its own.

You will load the hierarchy attribute from MilitaryVehicles as your ontology:

In [ ]:
import json

print(json.dumps(MilitaryVehicles.hierarchy, indent=2)[:900] + "\n...")

The hierarchy defines `land vehicle` along with `watercraft` and `aircraft`. When evaluated against this taxonomy,
unsampled concepts will be identified.

In [ ]:
from dataeval_flow.workflows.taxonomy import TaxonomyConfig

vehicle_ontology = MilitaryVehicles.hierarchy

vocab_workflow = TaxonomyConfig(name="vocab", ontology=vehicle_ontology)

task_vocab = TaskConfig(
    name="vehicles-vocab",
    workflow="vocab",
    sources="vehicles_src",  # the source scope read
)

config_vocab = PipelineConfig(
    max_processes=2,
    datasets=config_metadata.datasets,
    sources=config_metadata.sources,
    workflows=[vocab_workflow],
    tasks=[task_vocab],
)

result_vocab = run_task(config_vocab, task_vocab, cache_dir=Path("./cache"))
print(result_vocab.report())

The `representation` step's output is DataEval's collection worklist against the ontology, one row per leaf concept
short of its expected share. Its leaf coverage, total deficit and dark branches are attributes of it.

In [ ]:
representation = result_vocab.steps["representation"].output
print(f"leaf coverage: {representation.leaf_coverage:.1%}")
print(f"total deficit: {representation.total_deficit} labels\n")

print("Dark branches: whole regions of the label space with zero examples:")
for branch in representation.dark_branches.iter_rows(named=True):
    print(f"  {branch['label']:<16} {branch['leaves']} leaf class(es), zero examples")

print("\nTop of the collection worklist:")
for row in representation.data().head(8).iter_rows(named=True):
    print(f"  {row['label']:>14}  {row['action']:<8} have {row['count']:>4}, want {row['target']:>4}")

### What the ontology adds

`taxonomy` makes four findings:

- Leaf Coverage reports the leaf coverage (76.9%, 20 of the ontology's 26 leaves) and names the specific missing
  concepts. In addition to the four held-out Air Defense classes, it surfaces `aircraft` and `watercraft` concepts
  defined in the taxonomy. It warns: the coverage is under the default 0.9, and Air Defense is a wholly empty branch.
- Label Conformance verifies whether all collected dataset classes exist in the ontology. Each of the 24 class names
  resolves to one concept.
- Mergeability shows the mapping between dataset class names and ontology concepts, and provides relabeling rules
  where names differ. Here every class carries over one-to-one.
- Ontology Structure reports the concept hierarchy size, leaf counts, and depth: 34 concepts, 26 leaves, depth 4, and
  one single-child link, which it notes.

The two worklists differ in what they spread the 1,500 labels over. `scope`'s Class Shortfall spreads them over the 24
classes the dataset declares, a target of 62 each, 266 labels short. `taxonomy` spreads them over the ontology's 26
leaves, a target of 58 each, 358 labels short, and so adds `aircraft` and `watercraft`.

### Read the worklist with scope in mind

The report flags `Air Defense` as a dark branch (4 leaves, 0 samples), an in-scope gap that requires targeted data
collection.

The report also flags `aircraft` and `watercraft`. For a ground-vehicle system, these categories represent ontology
concepts outside operational scope. You should scope your ontology to match operational requirements so check
thresholds track valid system targets.

### When a label does not reconcile

Label conformance performs exact matching against ontology concepts. Any misspellings or unsanctioned classes are
flagged as unmatched:

In [ ]:
from dataeval import Ontology
from dataeval.core import label_reconciliation

# `taxonomy` builds this from its `ontology` field. You can also construct the
# object directly to reconcile arbitrary label lists against it.
ontology_obj = Ontology.from_hierarchy(vehicle_ontology)
check = label_reconciliation(["T-72", "BTR-80", "T72", "technical"], ontology_obj)
print("matched:  ", dict(check["matched"]))
print("unmatched:", list(check["unmatched"]))

`T-72` and `BTR-80` match successfully. `T72` fails due to the missing hyphen, and `technical` fails because it is
absent from the ontology.

### Share one ontology across tasks

You can define ontologies centrally under `ontologies:` in YAML and reference them by name. The presets run as tasks
on one source:

```yaml
ontologies:
  - name: vehicles
    source: config/vehicles.jsonld
    concepts:
      - id: http://example.org/vehicles#technical
        label: technical
        synonyms: [pickup-mounted, NSV]

workflows:
  - name: vocab
    type: taxonomy
    ontology: vehicles
  - name: bias
    type: bias
    metadata: vehicle_factors
  - name: coverage
    type: scope

tasks:
  - name: vehicles-vocab
    workflow: vocab
    sources: vehicles_src
  - name: vehicles-bias
    workflow: bias
    sources: vehicles_src
  - name: vehicles-coverage
    workflow: coverage
    sources: vehicles_src
```

## Run `scope` with an extractor

You can add an extractor to evaluate embedding coverage and dimensional completeness. To use a pretrained ONNX model
for the embeddings, see {doc}`onnx_embeddings`.

Embedding coverage identifies low-density or uncovered regions in feature space. Dimensional completeness evaluates
how uniformly samples span embedding dimensions.

You will configure a BoVW extractor with a 256-word vocabulary. The `isotropy` metric requires more samples per class
than embedding dimensions. For classes with fewer than 256 samples, isotropy reports `null`, while `dispersion` and
`near_duplicate_fraction` evaluate fully.

The `coverage` settings are the `coverage` step's. `dimensional-completeness` judges the completeness step, and
`class-coverage` reports the classes the `coverage` step flagged.

In [ ]:
from dataeval_flow.config.extractors import BoVWExtractorConfig

full_workflow = ScopeConfig.model_validate(
    {
        "name": "coverage-full",
        "coverage": {
            "method": "adaptive",
            "percent": 0.01,  # adaptive: flag the sparsest 1% of observations
            "num_observations": 50,  # Number of neighbors for coverage analysis
        },
        "checks": {
            "dimensional-completeness": {"warning": 0.5},  # Warn if completeness < 0.5
        },
    }
)

task_full = TaskConfig(
    name="vehicles-coverage-full",
    workflow="coverage-full",
    sources="vehicles_src",
    extractor="bovw_ext",
)

config_full = PipelineConfig(
    max_processes=2,
    seed=0,
    metadata=config_metadata.metadata,
    datasets=config_metadata.datasets,
    sources=config_metadata.sources,
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[full_workflow],
    tasks=[task_full],
)

In [ ]:
result_full = run_task(config_full, task_full, cache_dir=Path("./cache"))

### Full coverage report

Now the report judges Class Coverage and Dimensional Completeness beside the Class Shortfall. The Class Shortfall's
evidence repeats the report of the earlier `scope` run, so this cell prints the short form, `report(detailed=False)`,
which holds the summary of findings, the health and the Steps table. `report()` gives the full evidence. The cells
below read it from the steps.

In [ ]:
print(result_full.report(detailed=False))

### Inspect embedding results

The `coverage` step's output is DataEval's `Coverage` output: a table with one row per class, and the indices of the
uncovered items. The method is the entry's setting.

In [ ]:
import polars as pl

coverage = result_full.steps["coverage"].output
per_class = coverage.data()
total = int(per_class["count"].sum())
uncovered = len(coverage.uncovered_indices)
print(f"Method: {full_workflow.coverage.method}")
print(f"Uncovered: {uncovered} of {total} ({uncovered / total:.1%})\n")

columns = ["class", "count", "uncovered", "dispersion", "isotropy", "near_duplicate_fraction"]
flags = ["is_clustered", "is_one_dimensional", "has_near_duplicates"]
with pl.Config(tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True, float_precision=2):
    print(per_class.select(columns).sort("dispersion"))
    print(per_class.select("class", *flags).sort("class"))

### Read the per-class columns

Evaluate the per-class embedding metrics:

- `dispersion` is the class variance relative to the average class variance. Values near 1.0 indicate typical spread,
  and low values indicate tight clustering.
- `isotropy` is the directional variance across embedding dimensions. Low values indicate variation concentrated along
  few principal axes.
- `near_duplicate_fraction` is the proportion of samples whose nearest neighbor in the class is unusually close,
  judged against a typical class's spacing.

DataEval turns each into a verdict per class: `is_clustered` (dispersion under `dispersion_factor`, 0.5 by default),
`is_one_dimensional` (isotropy under `isotropy_factor` times the median class's, 0.5 by default) and
`has_near_duplicates` (near-duplicate fraction over `near_duplicate_threshold`, 0.1 by default). A verdict is `null`
where DataEval cannot judge, such as an isotropy it did not measure.

Across all 20 classes, dispersion is balanced (0.96 to 1.03) and near-duplicate fractions are 0.00, so the sampled
classes cover feature space well. Isotropy is `null` for every class, since none has more than 256 samples, and so is
`is_one_dimensional`. DataEval flags no class clustered or padded with near duplicates.

Adaptive coverage flags its `percent` of the items by construction, so 15 of 1,500 uncovered (1.0%) is the setting at
work rather than a finding. The 15 fall in 12 classes, no more than 2 in any one. The Class Coverage finding informs
while any item is uncovered, and warns only on a class DataEval flags clustered, one-dimensional or duplicate-padded:
there is none.

In [ ]:
completeness = result_full.steps["completeness"].output.data()
print("Dimensional Completeness:")
print(f"  Score: {completeness['completeness']:.3f}")
print(f"  Nearest neighbor pairs: {len(completeness['nearest_neighbor_pairs'])}")

Dimensional completeness is 0.572, between the warning band (0.5) and the info band (0.8), so its finding informs. On
visual variation, this sample shows no weakness the default thresholds warn on: its gap is in the taxonomy.

## Tune the check thresholds

Check thresholds control when a finding warns, as the Severity entry of the {doc}`../reference/glossary` describes.
Each preset keys its `checks` by check type, and the cutoffs that flag a class for `class-coverage` are the `coverage`
step's settings. The right thresholds depend on your domain:

| Preset | Check | Field | Default | Safety-critical | Web-scraped data |
|---|---|---|---|---|---|
| `scope` | `uncovered-items` | `warning` | 10% | 3–5% | 15–20% |
| `scope` | `dimensional-completeness` | `warning` | 0.5 | 0.7–0.8 | 0.3–0.4 |
| `bias` | `class-imbalance` | `warning` | 5:1 | 2–3:1 | 10–20:1 |
| `bias` | `factor-coverage-gaps` | `warning` | 2 | 1 | 5–10 |
| `scope` | `coverage` (step) | `dispersion_factor` | 0.5 | 0.7 | 0.3 |
| `scope` | `coverage` (step) | `isotropy_factor` | 0.5 | 0.7 | 0.3 |
| `scope` | `coverage` (step) | `near_duplicate_threshold` | 0.1 | 0.02 | 0.25 |
| `taxonomy` | `leaf-coverage` | `coverage` | 0.9 | 0.95 | 0.6 |
| `taxonomy` | `leaf-coverage` | `empty_branches` | 0 | 0 | 2–5 |
| `taxonomy` | `label-conformance` | `warning` | 0 | 0 | 3–10 |

`class-imbalance` (on `bias`) and `dimensional-completeness` also take `info`, the band between `ok` and a warning: a
ratio over 2.0, or a score under 0.8, informs by default. `null` turns a threshold off. The `coverage` factors are
DataEval's and take no `null`: each must be between 0 and 1, and isotropy is judged against the median class, not on
an absolute scale. `near_duplicate_threshold: 1` turns that flag off.

`uncovered-items` judges only `naive` coverage, since adaptive coverage flags its `percent` of the items by
construction. The bias and taxonomy thresholds go on their own entries, as `TaxonomyConfig(...,
checks={"leaf-coverage": {"coverage": 0.95}})`.

In [ ]:
from dataeval_flow.workflows.scope import ScopeChecks

strict_thresholds = ScopeChecks.model_validate({"dimensional-completeness": {"warning": 0.6}})
strict_coverage = full_workflow.coverage.model_copy(
    update={"dispersion_factor": 0.7, "isotropy_factor": 0.7, "near_duplicate_threshold": 0.02}
)

strict_workflow = full_workflow.model_copy(
    update={"name": "coverage-strict", "checks": strict_thresholds, "coverage": strict_coverage},
)

task_strict = TaskConfig(
    name="vehicles-coverage-strict",
    workflow="coverage-strict",
    sources="vehicles_src",
    extractor="bovw_ext",
)

config_strict = PipelineConfig(
    max_processes=2,
    seed=0,
    metadata=config_full.metadata,
    datasets=config_full.datasets,
    sources=config_full.sources,
    extractors=config_full.extractors,
    workflows=[full_workflow, strict_workflow],
    tasks=[task_strict],
)

result_strict = run_task(config_strict, task_strict, cache_dir=Path("./cache"))
print(result_strict.report(detailed=False))

Compare each finding's severity under the two sets of thresholds:

In [ ]:
for default, strict in zip(result_full.findings, result_strict.findings, strict=True):
    print(f"{default.title:<28} {default.severity:<8} -> {strict.severity:<8} {strict.brief}")

Only one additional warning triggers: Dimensional Completeness (0.572) falls below the strict 0.6 threshold. The
stricter `coverage` factors flag no class: no class's dispersion is under 0.7, every class's near-duplicate fraction
prints as 0.00, and isotropy is not measured.

You can adjust individual check thresholds to match your domain tolerance without altering underlying data
calculations. The `coverage` factors only change where DataEval draws its verdicts. `dispersion`, `isotropy` and
`near_duplicate_fraction` stay the same.

## Export the results

`result_full.export()` returns the result as a JSON string. See {doc}`../how_to/read_evaluation_outputs` for how to
read reports and result envelopes.

In [ ]:
json_str = result_full.export(fmt="json")
print(f"JSON output: {len(json_str)} characters")
print(json_str[:500] + "\n...")

## Conclusion

In this tutorial, you learned how to:

- Simulate missing categories using `Filter` view operations.
- Run `bias` to assess class balance, shortcut factors and cross-tabulated factor gaps, and `scope` without an
  extractor for the class worklist.
- Run `taxonomy` on the same source to benchmark the labels against an ontology and name missing categories.
- Reconcile label names against an ontology.
- Configure feature extractors to evaluate embedding dispersion and dimensional completeness.
- Read each step's output from `result.steps`.
- Tune check thresholds to enforce strict domain requirements.
- Export structured coverage results to JSON.
- Assess both label completeness against an ontology and feature diversity in embedding space to ensure comprehensive
  dataset coverage.

## Next steps

- {doc}`data_prioritization` — Feed coverage gap targets into a ranking that prioritizes acquisition of missing
  concepts.
- {doc}`drift_monitoring` — Track operational distribution shifts against a reference dataset.
- {doc}`../concepts/Coverage` — Learn how label-space and embedding-space coverage work.

## On your own

Repeat the analysis on your own data by changing these settings:

- Set `dataset` on the `DatasetProtocolConfig` to your own dataset in MAITE format. Change the classes that the
  `Filter` operation keeps, or remove it, to assess the whole collection.
- Set `intrinsic_factors` in the `vehicle_factors` metadata policy to the image statistics you want as factors. Set
  `exclude` to the metadata columns that name items instead of describing them, and `continuous_factor_bins` to the
  number of bins for each factor.
- Change `min_representation` in `factor-gaps`, which is 5 here, and `method` in `diversity`, which is `simpson` here.
- Change the values under `checks` of the `bias` entry and the `scope` entry, using the table in the "Tune the check
  thresholds" section.
- Change `method`, `percent` and `num_observations` in the `coverage` settings of `scope`. They are `adaptive`, 0.01
  and 50 here.
- Change `vocab_size` on `BoVWExtractorConfig`, which is 256 here, or replace the extractor config.
- Set `ontology` on `TaxonomyConfig` to your own hierarchy, or in YAML to the name of an `ontologies:` entry.